<a href="https://colab.research.google.com/github/darrickpang/Email/blob/master/Copy_of_NMT_project_Darrick_Pang.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
import time
import pandas as pd
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from datasets import load_dataset

In [ ]:
results = []
epochs = 10
batch_size = 256
training_samples = 3000000
model = "Transformer"
range_bleu = 1000

source_vocab = 30000
target_vocab = 30000
embedding_dim = 256
latent_dim = 512

# ---- Hyperparams (small, fast) ----
d_model = 512          # model width
num_heads = 8
d_ff = 1024            # FFN hidden
num_enc = 3            # encoder layers
num_dec = 3            # decoder layers
dropout_rate = 0.1
v_src = source_vocab   # 30_000 from your code
v_tgt = target_vocab   # 30_000 from your code


In [ ]:
# pip install evaluate

In [ ]:
# pip install sacrebleu

In [ ]:
# from evaluate import load
# bleu = load("sacrebleu")

In [ ]:
dataset = load_dataset("wmt16", "de-en")

train_data = dataset["train"].select(range(training_samples))
test_data = dataset["test"]
val_data = dataset["validation"]

In [ ]:
print(train_data["translation"])
print(val_data)

In [ ]:
source_text = [german["de"] for german in train_data["translation"]]
# print(source_text)

target_text = [english["en"] for english in train_data["translation"]]
# print(target_text)

# Add start and end tokens to target text
target_text_with_tokens = ['<start> ' + text + ' <end>' for text in target_text]


source_tokenizer = Tokenizer(num_words=30000, filters='')
target_tokenizer = Tokenizer(num_words=30000, filters='')

source_tokenizer.fit_on_texts(source_text)
target_tokenizer.fit_on_texts(target_text_with_tokens)

source_sequence = source_tokenizer.texts_to_sequences(source_text)
target_sequence = target_tokenizer.texts_to_sequences(target_text_with_tokens)

max_src_len = 40
max_tgt_len = 40
encoder_input = pad_sequences(source_sequence, maxlen=max_src_len, padding='post')
decoder_input = pad_sequences([s[:-1] for s in target_sequence], maxlen=max_tgt_len, padding='post')
decoder_target = pad_sequences([s[1:] for s in target_sequence], maxlen=max_tgt_len, padding='post')

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

# ---- Positional Encoding ----
class PositionalEncoding(layers.Layer):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        import numpy as np
        pe = np.zeros((max_len, d_model), dtype="float32")
        position = np.arange(0, max_len)[:, None]
        div = np.exp(np.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = np.sin(position * div)
        pe[:, 1::2] = np.cos(position * div)
        self.pe = tf.constant(pe[None, ...])   # [1, max_len, d_model]
    def call(self, x):
        return x + self.pe[:, :tf.shape(x)[1], :]

# ---- Masks ----
def padding_mask(x):
    # x: [B, T] int32
    return tf.cast(tf.equal(x, 0), tf.bool)  # True where PAD

def causal_mask(T):
    return tf.linalg.band_part(tf.ones((T, T), dtype=tf.bool), -1, 0)  # lower-triangular True

# ---- Encoder/Decoder Blocks ----
def encoder_block(x, pad_mask):
    # x: [B, T, d_model]
    attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model, dropout=dropout_rate)
    y = attn(query=x, value=x, key=x, attention_mask=~pad_mask[:, None, None, :])  # mask True=keep
    x = layers.LayerNormalization(epsilon=1e-6)(x + layers.Dropout(dropout_rate)(y))
    y = layers.Dense(d_ff, activation="relu")(x)
    y = layers.Dense(d_model)(y)
    x = layers.LayerNormalization(epsilon=1e-6)(x + layers.Dropout(dropout_rate)(y))
    return x

def decoder_block(x, enc_out, look_mask, enc_pad_mask):
    # 1️⃣ Self-attention
    self_attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model, dropout=dropout_rate)
    y = self_attn(query=x, value=x, key=x, attention_mask=look_mask[:, None, :, :])
    x = layers.LayerNormalization(epsilon=1e-6)(x + layers.Dropout(0.1)(y))

    # 2️⃣ Cross-attention (encoder–decoder)
    cross_attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model, dropout=dropout_rate)
    y = cross_attn(query=x, value=enc_out, key=enc_out, attention_mask=~enc_pad_mask[:, None, None, :])
    x = layers.LayerNormalization(epsilon=1e-6)(x + layers.Dropout(0.1)(y))

    # 3️⃣ Feed-forward
    y = layers.Dense(d_ff, activation="relu")(x)
    y = layers.Dense(d_model)(y)
    x = layers.LayerNormalization(epsilon=1e-6)(x + layers.Dropout(0.1)(y))
    return x


# ---- Inputs (reuse your max_src_len / max_tgt_len) ----
enc_inp = layers.Input(shape=(max_src_len,), name="enc_tokens")
dec_inp = layers.Input(shape=(max_tgt_len,), name="dec_tokens")  # teacher-forced full sequence

# Embeddings (+ tie dims)
enc_emb = layers.Embedding(v_src, d_model, mask_zero=True)(enc_inp)
dec_emb = layers.Embedding(v_tgt, d_model, mask_zero=True)(dec_inp)

# Add positional encodings
enc_x = PositionalEncoding(d_model)(enc_emb)
dec_x = PositionalEncoding(d_model)(dec_emb)

# Masks
enc_pad = layers.Lambda(lambda x: tf.cast(tf.equal(x, 0), tf.bool), name="enc_pad")(enc_inp)

# Decoder look-ahead + padding mask
def make_lookahead_mask(x):
    seq_len = tf.shape(x)[1]
    mask = tf.cast(tf.not_equal(x, 0), tf.bool)
    mask = tf.logical_and(tf.tile(mask[:, None, :], [1, seq_len, 1]),
                          tf.linalg.band_part(tf.ones((seq_len, seq_len), dtype=tf.bool), -1, 0))
    return mask

look = layers.Lambda(make_lookahead_mask, name="lookahead_mask")(dec_inp)

# Encoder stack
for _ in range(num_enc):
    enc_x = encoder_block(enc_x, enc_pad)

# Decoder stack
x = dec_x
for _ in range(num_dec):
    x = decoder_block(x, enc_x, look, enc_pad)

# Output projection
logits = layers.Dense(v_tgt, activation="softmax")(x)

transformer = Model([enc_inp, dec_inp], logits)


In [ ]:
# Label smoothing (improves BLEU a bit)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=False,
    # label_smoothing=0.1 # Removed unsupported argument
)

from tensorflow.keras import backend as K

def smoothed_sparse_categorical_crossentropy(y_true, y_pred, label_smoothing=0.1):
    y_true = tf.cast(y_true, tf.int32)
    num_classes = tf.shape(y_pred)[-1]
    y_true_one_hot = tf.one_hot(y_true, depth=num_classes)
    smooth_positives = 1.0 - label_smoothing
    smooth_negatives = label_smoothing / tf.cast(num_classes, tf.float32)
    y_true_smooth = y_true_one_hot * smooth_positives + smooth_negatives
    loss = -tf.reduce_sum(y_true_smooth * tf.math.log(y_pred + 1e-7), axis=-1)
    return tf.reduce_mean(loss)

# Noam-style schedule (Transformer baseline)
class NoamSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, d_model, warmup_steps=4000):
        self.d_model = tf.cast(d_model, tf.float32)
        self.warmup_steps = warmup_steps
    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        return (self.d_model ** -0.5) * tf.minimum(step ** -0.5, step * (self.warmup_steps ** -1.5))

lr = NoamSchedule(d_model, warmup_steps=4000)
opt = tf.keras.optimizers.Adam(learning_rate=lr, beta_1=0.9, beta_2=0.98, epsilon=1e-9)

transformer.compile(optimizer=opt, loss=lambda y_true, y_pred: smoothed_sparse_categorical_crossentropy(y_true, y_pred, label_smoothing=0.1), metrics=["accuracy"])

In [ ]:
start_time = time.time()

early_stop = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
# reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, verbose=1)

transformer.fit(
    [encoder_input, decoder_input],
    decoder_target,
    batch_size=batch_size,
    epochs=epochs,
    validation_split=0.1,
    callbacks=[early_stop]
)

end_time = time.time()
elapsed = end_time - start_time
print(f"Training time: {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")

Epoch 1/40
7032/7032 ━━━━━━━━━━━━━━━━━━━━ 1692s 232ms/step - accuracy: 0.5363 - loss: 4.5497 - val_accuracy: 0.6689 - val_loss: 3.2905
Epoch 2/40
7032/7032 ━━━━━━━━━━━━━━━━━━━━ 1587s 226ms/step - accuracy: 0.7174 - loss: 2.7081 - val_accuracy: 0.6900 - val_loss: 3.1393
Epoch 3/40
7032/7032 ━━━━━━━━━━━━━━━━━━━━ 1587s 226ms/step - accuracy: 0.7403 - loss: 2.5567 - val_accuracy: 0.6974 - val_loss: 3.0781
Epoch 4/40
7032/7032 ━━━━━━━━━━━━━━━━━━━━ 1588s 226ms/step - accuracy: 0.7504 - loss: 2.4910 - val_accuracy: 0.7019 - val_loss: 3.0434
Epoch 5/40
7032/7032 ━━━━━━━━━━━━━━━━━━━━ 1587s 226ms/step - accuracy: 0.7567 - loss: 2.4510 - val_accuracy: 0.7038 - val_loss: 3.0317
Epoch 6/40
7032/7032 ━━━━━━━━━━━━━━━━━━━━ 1589s 226ms/step - accuracy: 0.7612 - loss: 2.4222 - val_accuracy: 0.7063 - val_loss: 3.0097
Epoch 7/40
7032/7032 ━━━━━━━━━━━━━━━━━━━━ 1587s 226ms/step - accuracy: 0.7645 - loss: 2.4017 - val_accuracy: 0.7078 - val_loss: 3.0029
Epoch 8/40
7032/7032 ━━━━━━━━━━━━━━━━━━━━ 1588s 226ms/s

In [ ]:
def translate(sentence, beam_width=4, max_len=max_tgt_len, alpha=0.6):
    # ---- Encode the source sentence ----
    src_seq = source_tokenizer.texts_to_sequences([sentence])
    src_seq = pad_sequences(src_seq, maxlen=max_src_len, padding='post')

    start_id = target_tokenizer.word_index['<start>']
    end_id   = target_tokenizer.word_index['<end>']

    # Each beam is (sequence_so_far, cumulative_log_prob)
    beams = [([start_id], 0.0)]

    for _ in range(max_len):
        new_beams = []
        for seq, score in beams:
            # Stop expanding finished hypotheses
            if seq[-1] == end_id:
                new_beams.append((seq, score))
                continue

            dec_seq = pad_sequences([seq], maxlen=max_tgt_len, padding='post')
            preds = transformer.predict([src_seq, dec_seq], verbose=0)
            probs = preds[0, len(seq)-1, :]  # distribution for next token

            # pick top-k candidates
            top_ids = np.argsort(probs)[-beam_width:]
            for t in top_ids:
                new_seq = seq + [int(t)]
                new_score = score + np.log(probs[t] + 1e-9)
                new_beams.append((new_seq, new_score))

        # Keep the best `beam_width` beams
        # ---- Length normalization function ----
        def length_norm(score, length, alpha=alpha):
            return score / ((5 + length) / 6) ** alpha

        # Keep the best normalized beams
        beams = sorted(
            new_beams,
            key=lambda x: length_norm(x[1], len(x[0])),
            reverse=True
        )[:beam_width]

        # Early-stop if all beams ended
        if all(seq[-1] == end_id for seq, _ in beams):
            break

    # Take best-scoring beam
    best_seq = beams[0][0]
    words = [target_tokenizer.index_word.get(i, '') for i in best_seq[1:] if i not in (0, end_id)]
    return ' '.join(words)

In [ ]:
print(translate("Das ist ein Test."))

that is a


In [ ]:
# Generate predictions on a small subset of validation data
predictions = []
references = []

for i in range(range_bleu):  # 200 sentences for demo, increase later
    de_sentence = test_data[i]["translation"]["de"]
    en_reference = test_data[i]["translation"]["en"]

    en_predicted = translate(de_sentence, beam_width=6, alpha=0.7)

    predictions.append(en_predicted)
    references.append([en_reference])  # sacreBLEU expects list of list

result = bleu.compute(predictions=predictions, references=references)
print(f"BLEU score: {result['score']:.2f}")

BLEU score: 13.97


In [ ]:
results.append({"Epochs": epochs, "Batch Size": batch_size, "Training Time": elapsed, "BLEU": result['score'], "Training Data": training_samples, "Model": model, "Range BLEU": range_bleu})

In [ ]:
df = pd.DataFrame(results)
df

In [ ]:
df.to_excel('NMT_output_Darrick_Pang.xlsx', sheet_name='MyData')

In [ ]:
import pandas as pd
df = pd.read_excel('NMT_output_Darrick_Pang.xlsx')
df

,Unnamed: 0,Epochs,Batch Size,Training Time (seconds),BLEU,Training Data,Model,Range BLEU
0,0,10,64,3076.700000,2.850000,50000,LSTM,2
1,1,1,32,1054.220000,6.340000,100000,LSTM,2
2,2,1,32,621.710000,8.550000,100000,Bidirectional LSTM,2
3,3,10,32,6146.566196,4.891851,100000,Bidirectional LSTM,10
4,4,10,32,6146.566196,3.381288,100000,Bidirectional LSTM,200
5,5,15,64,1287.262952,6.004166,200000,Transformer,200
6,6,15,64,5725.310183,9.404734,1000000,Transformer,200
7,7,15,64,5312.574600,8.750748,1000000,Transformer,200
8,8,15,64,5415.191232,9.629194,1000000,Transformer,200
9,9,20,64,6993.629845,8.974931,1000000,Transformer,200
